In [2]:
import glob
import os
import sys
from pathlib import Path
from typing import Union
from typing import Optional
from pandas import DataFrame
import pandas as pd
from PIL import Image
from utills import (
    find_all_predicted_tomo_id_attributes,
    find_all_tomo_id_attributes,
    find_tomo_id_slice_attributes,
)

# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config

In [ ]:
root_dir = "/data/horse/ws/kein254g-team_project/rtdetr_batch/exp2"
labels_dir = f"{root_dir}/labels"
images_dir = "/data/horse/ws/kein254g-team_project/test_flat"
txt_files = glob.glob(f"{labels_dir}/*.txt")
test_image_files = glob.glob(f"{images_dir}/*.jpg")

label_stems = {Path(p).stem for p in txt_files}
unlabeled_images = [p for p in test_image_files if Path(p).stem not in label_stems]

rows = []
print(len(txt_files))


def get_image_width_height(image_path, file_name):
    file_name = os.path.splitext(file_name)[0] + ".jpg"
    image_path = os.path.join(images_dir, f"{file_name}")

    with Image.open(image_path) as img:
        return img.size  # returns (width, height)


for file_path in txt_files:
    print(f"\n--- Reading: {file_path} ---")
    with open(file_path, encoding="utf-8") as f:
        file_name = os.path.basename(file_path)  # e.g. "image1.txt"
        file_id = os.path.splitext(file_name)[0]  # e.g. "image1"
        _, id, _, z = file_id.split("_")
        cls, confidence, x_center, y_center, width, height = f.readline().strip().split()
        id = "tomo_" + str(id)

        W, H = get_image_width_height(images_dir, file_name)
        print(f"Tomo Id: {id} Image Width: {W}, Height: {H}")

        # denormalize
        x = int(round(float(x_center) * W))
        y = int(round(float(y_center) * H))

        rows.append(
            {
                "tomo_id": id,  # just the name without extension
                "Motor axis 0": int(z),  # full .txt file name
                "Motor axis 1": y,
                "Motor axis 2": x,
            }
        )

for img_path in unlabeled_images:
    file_name = os.path.basename(img_path)
    file_id = os.path.splitext(file_name)[0]
    _, id, _, z = file_id.split("_")

    id = "tomo_" + str(id)
    print(f"Tomo Id: {id} Image Width: {W}, Height: {H}")

    rows.append(
        {
            "tomo_id": id,
            "Motor axis 0": -1,
            "Motor axis 1": -1,
            "Motor axis 2": -1,
        }
    )


df = pd.DataFrame(rows)
df.to_csv("./submission.csv", index=False)
print("CSV created: my_submisson.csv")

46

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_00e047_slice_0170.txt ---
Tomo Id: tomo_00e047 Image Width: 928, Height: 959

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0151.txt ---
Tomo Id: tomo_01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0139.txt ---
Tomo Id: tomo_01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0146.txt ---
Tomo Id: tomo_01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_00e047_slice_0176.txt ---
Tomo Id: tomo_00e047 Image Width: 928, Height: 959

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0153.txt ---
Tomo Id: tomo_01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_projec

In [ ]:
def ground_truth_distance(tomoId, slice) -> Union[float, None]:
    row_gt_df: Optional[DataFrame] = find_tomo_id_slice_attributes(tomoId, slice)
    row_predicted_df: Optional[DataFrame] = find_all_predicted_tomo_id_attributes(tomoId)

    if row_gt_df is None or row_predicted_df is None:
        return None

    row_gt = row_gt_df.iloc[0]
    row_predicted = row_predicted_df.iloc[0]
    voxel_spacing = row_gt["Voxel spacing"]

    gt_x = row_gt["Motor axis 2"]
    gt_y = row_gt["Motor axis 1"]
    gt_z = row_gt["Motor axis 0"]

    pred_x = row_predicted["x_center"]
    pred_y = row_predicted["y_center"]
    pred_z = row_predicted["slice"]

    # converts pixel values to physical angstroms unit
    diff_x = (gt_x - pred_x) * voxel_spacing
    diff_y = (gt_y - pred_y) * voxel_spacing
    diff_z = (gt_z - pred_z) * voxel_spacing

    distance = (diff_x**2 + diff_y**2 + diff_z**2) ** 0.5
    return float(distance)


def pick_max_confidence(tomoId):
    rows = find_all_tomo_id_attributes(tomoId)
    max_confidence = -1
    max_row = None

    if rows is None or rows.empty:
        return None

    for index, row in rows.iterrows():
        confidence = float(row["confidence"])
        if confidence > max_confidence:
            max_confidence = confidence
            max_row = row

    return max_row


DISTANCE_THRESHOLD = 30.0  # in angstroms


def cheap_NMS():
    predictions_df = pd.read_csv(config.DENORMALIZED_RESULTS)
    predictions_df = predictions_df.sort_values(by="confidence", ascending=False)

    nms_rows = []
    for tomo_id, group in predictions_df.groupby("tomo_id"):
        selected = []
        for idx, row in group.iterrows():
            # Check if this prediction is sufficiently far from all previously selected predictions
            too_close = False
            for sel_row in selected:
                # Calculate distance in (x_center, y_center, slice) space
                dx = row["x_center"] - sel_row["x_center"]
                dy = row["y_center"] - sel_row["y_center"]
                dz = row["slice"] - sel_row["slice"]
                dist = (dx**2 + dy**2 + dz**2) ** 0.5
                if dist < DISTANCE_THRESHOLD:
                    too_close = True
                    break
            if not too_close:
                selected.append(row)
        nms_rows.extend(selected)

    nms_df = pd.DataFrame(nms_rows)
    nms_df.to_csv(config.NMS_RESULTS_CSV, index=False)

    print(nms_df)


cheap_NMS()

        tomo_id  slice  confidence   x_center   y_center      width  \
13  tomo_00e047    180    0.653145  541.68768  65.217984  63.911424   
27  tomo_01a877    142    0.311191  619.56528  62.592115  60.600070   

        height  
13  826.890688  
27  846.459424  
